# Analysis of threats to Jamaica's natural ecosystems

## Step 0: import relevant packages and set up base path and crs

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import fiona
from pathlib import Path
import matplotlib.pyplot as plt
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
import os
from rasterio.features import geometry_mask
from rasterio.transform import from_origin
from rasterio.windows import Window
from rasterio.plot import show
import scipy.ndimage as nd
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

In [ ]:
import matplotlib
print(matplotlib.rcParams['font.family'])
matplotlib.rcParams['font.family'] = 'Times New Roman'

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Inputs")

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
jamaica_boundary_path = base_path / "Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

In [ ]:
# Paths to protected areas shapefiles
forest_reserves_path = base_path / "protected_landcover/Forest_reserves.shp"
protected_areas_path = base_path / "protected_landcover/Protected_areas.shp"

# Load the protected areas shapefiles
forest_reserves = gpd.read_file(forest_reserves_path)
protected_areas = gpd.read_file(protected_areas_path)

# Reproject both layers to the same CRS as the landcover and bauxite reserves
forest_reserves = forest_reserves.to_crs(jamaica_metric_grid_crs)
protected_areas = protected_areas.to_crs(jamaica_metric_grid_crs)

# Combine the protected areas using a union operation
combined_protected_layers = gpd.overlay(forest_reserves, protected_areas, how='union')

# Inspect the combined protected layers to verify
print(combined_protected_layers.head())
print(combined_protected_layers.crs)

In [ ]:
land_use = base_path / "2013_landuse_LandCover.shp"

In [ ]:
terrestrial_landcover = gpd.read_file(land_use)
terrestrial_landcover = terrestrial_landcover.to_crs(jamaica_metric_grid_crs)
print(terrestrial_landcover.crs)

In [ ]:
categories = terrestrial_landcover['Classify'].unique()
# Display the categories
print("Land Use Categories:")
for category in categories:
    print("-", category)

In [ ]:
# Calculate area in square meters
terrestrial_landcover['area_m2'] = terrestrial_landcover.geometry.area

# Group by 'Classify' and sum the areas
area_by_category = terrestrial_landcover.groupby('Classify')['area_m2'].sum().reset_index()

# Display the results
#print(area_by_category)

# Calculate total area
total_area = area_by_category['area_m2'].sum()
print(f"Total Area: {total_area:.2f} square meters")

# Calculate percentage for each category
area_by_category['percentage'] = (area_by_category['area_m2'] / total_area) * 100

# Sort the DataFrame by percentage in descending order
area_by_category = area_by_category.sort_values(by='percentage', ascending=False)

# Display the updated DataFrame
display(area_by_category)

In [ ]:
# Format the DataFrame for better readability
area_by_category['area_km2'] = area_by_category['area_m2'] / 1e6  # Convert area to square kilometers
area_by_category['percentage'] = area_by_category['percentage'].round(2)  # Round percentage to 2 decimal places

# Select and reorder columns
area_summary = area_by_category[['Classify', 'area_km2', 'percentage']]

# Rename columns for clarity
area_summary.columns = ['Land Use Category', 'Area (km²)', 'Percentage of Total Area (%)']

# Display the summary table
display(area_summary)

In [ ]:
# Example mapping dictionary
landuse_category_mapping = {
    'Bare Rock': 'Bare Rock',
    'Fields: Herbaceous crops, fallow, cultivated vegetables': 'Agriculture',
    'Fields: Pasture,Human disturbed, grassland': 'Agriculture',
    'Herbaceous Wetland': 'Freshwater wetland',
    'Mangrove Forest': 'Mangrove',
    'Fields: Bare Land': 'Agriculture',
    'Open dry forest - Short': 'Open dry forest',
    'Open dry forest - Tall (Woodland/Savanna)': 'Open dry forest',
    'Plantation: Tree crops, shrub crops, sugar cane, banana': 'Plantation',
    'Quarry': 'Bauxite extraction / quarry',
    'Water Body': 'Water body',
    'Buildings and other infrastructures': 'Buildings and other infrastructure',
    'Fields and Secondary Forest': 'Mixed land use: forests with bamboo or agriculture/plantation',
    'Bamboo and Fields': 'Mixed land use: agriculture and bamboo',
    'Bauxite Extraction': 'Bauxite extraction / quarry',
    'Disturbed broadleaved forest (Secondary Forest)': 'Forest',
    'Fields  and Bamboo': 'Mixed land use: agriculture and bamboo',
    'Bamboo and Secondary Forest': 'Mixed land use: agriculture and bamboo',
    'Hardwood Plantation: Euculytus': 'Plantation',
    'Hardwood Plantation: Mixed': 'Plantation',
    'Swamp Forest': 'Swamp forest',
    'Fields or Secondary Forest/Pine Plantation': 'Mixed land use: forests with bamboo or agriculture/plantation',
    'Hardwood Plantation: Mahoe': 'Plantation',
    'Hardwood Plantation: Mahogany': 'Plantation',
    'Bamboo': 'Bamboo',
    'Closed broadleaved forest (Primary Forest)': 'Forest',
    'Secondary Forest': 'Forest'
    # Add other mappings as needed
}


# Map the categories
terrestrial_landcover['Classify'] = terrestrial_landcover['Classify'].replace(landuse_category_mapping)

In [ ]:
# Calculate area in square meters
terrestrial_landcover['area_m2'] = terrestrial_landcover.geometry.area

# Handle invalid geometries if necessary
# terrestrial_landcover['geometry'] = terrestrial_landcover['geometry'].buffer(0)

# Drop rows with missing 'Classify' values
terrestrial_landcover = terrestrial_landcover.dropna(subset=['Classify'])

# Group by 'Classify' and sum the areas
area_by_category = terrestrial_landcover.groupby('Classify')['area_m2'].sum().reset_index()

# Calculate total area
total_area = area_by_category['area_m2'].sum()

# Calculate percentage for each category
area_by_category['percentage'] = (area_by_category['area_m2'] / total_area) * 100

# Convert area to square kilometers
area_by_category['area_km2'] = area_by_category['area_m2'] / 1e6

# Round the percentage
area_by_category['percentage'] = area_by_category['percentage'].round(2)

# Select and reorder columns
area_summary = area_by_category[['Classify', 'area_km2', 'percentage']]

# Rename columns for clarity
area_summary.columns = ['Land Use Category', 'Area (km²)', 'Percentage of Total Area (%)']

# Display the summary table
display(area_summary)


###  Urban and agricultural encroachment

In [ ]:
# Encroachment on mangroves

# Step 1: Create a 100m buffer around mangroves
mangrove_buffer = terrestrial_landcover[terrestrial_landcover['Classify'] == 'Mangrove'].copy()
mangrove_buffer['geometry'] = mangrove_buffer.geometry.buffer(100)

In [ ]:
# Step 2: Intersect the buffer with the land use layer
buffered_landuse_intersection = gpd.overlay(terrestrial_landcover, mangrove_buffer, how='intersection')

In [ ]:
# Step 3: Calculate area for each intersected land use
buffered_landuse_intersection['area_m2'] = buffered_landuse_intersection.geometry.area

In [ ]:
buffered_landuse_intersection.head()

In [ ]:
# Step 4: Group by land use type to calculate total area within the buffer
encroachment_risk_summary = (
    buffered_landuse_intersection.groupby('Classify_1')['area_m2']
    .sum()
    .reset_index()
)

In [ ]:
# Convert area to square kilometers and add a percentage column
encroachment_risk_summary['area_km2'] = encroachment_risk_summary['area_m2'] / 1e6
encroachment_risk_summary['percentage'] = (
    encroachment_risk_summary['area_m2'] / encroachment_risk_summary['area_m2'].sum()
) * 100

# Display the summary
display(encroachment_risk_summary)

# Optional: Save the results to a CSV file
# encroachment_risk_summary.to_csv("mangrove_encroachment_risk.csv", index=False)

In [ ]:
# Ensure the color mapping is applied to the buffered_landuse_intersection GeoDataFrame
buffered_landuse_intersection['color'] = buffered_landuse_intersection['Classify_1'].map(category_colors)

# Plot the buffered_landuse_intersection with the assigned colors
fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot the intersected land use areas within the mangrove buffer
buffered_landuse_intersection.plot(
    ax=ax,
    color=buffered_landuse_intersection['color'],
    edgecolor='black',
    linewidth=0.1
)

# Plot the original mangrove areas for reference
mangrove_buffer.plot(
    ax=ax,
    facecolor='none',
    edgecolor='blue',
    linestyle='--',
    linewidth=1,
    label='Mangrove Buffer (100m)'
)

# Plot the Jamaica boundary for context
jamaica_boundary.plot(
    ax=ax,
    facecolor='none',
    edgecolor='black',
    linewidth=1,
    linestyle='--',
    label='Jamaica Boundary'
)

# Prepare legend handles for land use types
legend_handles = []
for land_use in encroachment_risk_summary['Classify_1']:
    color = category_colors.get(land_use, 'grey')  # Use grey as a fallback color
    label = f"{land_use} ({encroachment_risk_summary[encroachment_risk_summary['Classify_1'] == land_use]['percentage'].values[0]:.2f}%)"
    patch = mpatches.Patch(color=color, label=label)
    legend_handles.append(patch)

# Add legend
legend = ax.legend(
    handles=legend_handles,
    title="Land Use in 100m Mangrove Buffer",
    bbox_to_anchor=(0.5, -0.1),
    loc='upper center',
    ncol=3,
    frameon=False,
    fontsize=12,
    title_fontsize=14,
    labelspacing=1.0,
    prop={'family': 'Times New Roman'}
)

# Add north arrow and scale bar
add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03)
add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, label_offset=0.04, km_offset=0.01)

# Add title
plt.title(
    "Land Use Within 100m Buffer of Mangroves",
    fontsize=20,
    fontweight='bold',
    fontname='Times New Roman',
    loc='center',
    pad=20
)

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
# Step 1: Create a 100m buffer around mangroves
mangrove_buffer = terrestrial_landcover[terrestrial_landcover['Classify'] == 'Mangrove'].copy()
mangrove_buffer['geometry'] = mangrove_buffer.geometry.buffer(100)

# Step 2: Exclude mangroves from the terrestrial landcover
non_mangrove_landcover = terrestrial_landcover[terrestrial_landcover['Classify'] != 'Mangrove']

# Step 3: Intersect the buffer with the non-mangrove land use layer
buffered_landuse_intersection = gpd.overlay(non_mangrove_landcover, mangrove_buffer, how='intersection')

# Step 4: Calculate area for each intersected land use
buffered_landuse_intersection['area_m2'] = buffered_landuse_intersection.geometry.area

# Display the results
buffered_landuse_intersection.head()

# Step 5: Group by land use type to calculate total area within the buffer
encroachment_risk_summary = (
    buffered_landuse_intersection.groupby('Classify_1')['area_m2']
    .sum()
    .reset_index()
)

# Convert area to square kilometers and add a percentage column
encroachment_risk_summary['area_km2'] = encroachment_risk_summary['area_m2'] / 1e6
encroachment_risk_summary['percentage'] = (
    encroachment_risk_summary['area_m2'] / encroachment_risk_summary['area_m2'].sum()
) * 100

# Display the summary
display(encroachment_risk_summary)

# Optional: Save the results to a CSV file
# encroachment_risk_summary.to_csv("mangrove_encroachment_risk.csv", index=False)

# Map the results with land use colors
buffered_landuse_intersection['color'] = buffered_landuse_intersection['Classify_1'].map(category_colors)

# Plot the results
fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot the intersected land use areas within the mangrove buffer
buffered_landuse_intersection.plot(
    ax=ax,
    color=buffered_landuse_intersection['color'],
    edgecolor='black',
    linewidth=0.1
)

# Plot the mangrove buffer
mangrove_buffer.plot(
    ax=ax,
    facecolor='none',
    edgecolor='blue',
    linestyle='--',
    linewidth=1,
    label='Mangrove Buffer (100m)'
)

# Plot the Jamaica boundary
jamaica_boundary.plot(
    ax=ax,
    facecolor='none',
    edgecolor='black',
    linewidth=1,
    linestyle='--',
    label='Jamaica Boundary'
)

# Add the legend
legend_handles = []
for _, row in encroachment_risk_summary.iterrows():
    land_use = row['Classify']
    color = category_colors.get(land_use, 'grey')
    percentage = row['percentage']
    label = f"{land_use} ({percentage:.2f}%)"
    patch = mpatches.Patch(color=color, label=label)
    legend_handles.append(patch)

legend = ax.legend(
    handles=legend_handles,
    title="Land Use in 100m Mangrove Buffer",
    bbox_to_anchor=(0.5, -0.1),
    loc='upper center',
    ncol=3,
    frameon=False,
    fontsize=12,
    title_fontsize=14,
    labelspacing=1.0,
    prop={'family': 'Times New Roman'}
)

# Add north arrow and scale bar
add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03)
add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, label_offset=0.04, km_offset=0.01)

# Add title
plt.title(
    "Land Use Within 100m Buffer of Mangroves (Excluding Mangroves)",
    fontsize=20,
    fontweight='bold',
    fontname='Times New Roman',
    loc='center',
    pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
Encroachment on forests

### Bauxite mining

In [ ]:
# Land use on bauxite reserves

In [ ]:
# Define the path to your nsdmb geodatabase
nsdmb_path = base_path / "nsdmb/GWP_Jamaica_NSP_Master_Geodatabase_v01.gdb"

In [ ]:
# List all layers in the geodatabase
layers = fiona.listlayers(nsdmb_path)
#("Available layers in the geodatabase:")
#for layer in layers:
    #print(layer)

In [ ]:
# Choose a layer to load (replace 'your_layer_name' with the actual layer name)
layer_name = "bauxite_reserves"

# Read the layer into a GeoDataFrame
bauxite_reserves = gpd.read_file(nsdmb_path, layer=layer_name)

# Inspect the first few rows
display(bauxite_reserves.head())

bauxite_reseves = bauxite_reserves.to_crs(jamaica_metric_grid_crs)
print(bauxite_reserves.crs)

In [ ]:
# Plot bauxite reserves
#gdf.plot(color="grey", edgecolor="black", figsize=(10, 10))
#plt.title("Bauxite Reserves")
#plt.show()

# Create a base plot for the Jamaica boundary
fig, ax = plt.subplots(figsize=(10, 10))
jamaica_boundary.plot(ax=ax, color="none", edgecolor="black", linewidth=1, label="Jamaica Boundary")

# Overlay the bauxite reserves
bauxite_reserves.plot(ax=ax, color="grey", edgecolor="black", alpha=0.7, label="Bauxite Reserves")

# Add a title and legend
plt.title("Bauxite Reserves Over Jamaica Boundary")
plt.legend()
plt.show()


In [ ]:
# Perform intersection of bauxite reserves and land use
landcover_bauxite = gpd.overlay(bauxite_reserves, terrestrial_landcover, how="intersection")

# Inspect the result
print(landcover_bauxite.head())

In [ ]:
landcover_bauxite.plot()
display(landcover_bauxite.head())

In [ ]:
# Calculate the area of each intersected polygon
landcover_bauxite['area_m2'] = landcover_bauxite.geometry.area

# Aggregate by land use type
bauxite_area_summary = (
    landcover_bauxite.groupby('Classify')['area_m2']
    .sum()
    .reset_index()
)

# Calculate total area on bauxite reserves
total_bauxite_area = bauxite_area_summary['area_m2'].sum()

# Calculate percentage of each land use type on bauxite reserves
bauxite_area_summary['percentage'] = (bauxite_area_summary['area_m2'] / total_bauxite_area) * 100

# Add area in km²
bauxite_area_summary['area_km2'] = bauxite_area_summary['area_m2'] / 1e6

# Rename columns for clarity
bauxite_area_summary.columns = ['Land Use Type', 'Area (m²)', 'Percentage (%)', 'Area (km²)']

# Display summary
display(bauxite_area_summary)


In [ ]:
# Percentage land use area on bauxite

In [ ]:
custom_colors = {
    'Bare Rock': '#A9A9A9',  # Dark Gray (rocky terrain)
    'Agriculture': '#8B4513',  # Dark Brown
    'Freshwater wetland': '#4169E1', #Royal blue
    'Mangrove': '#008080', # Teal Blue
    'Open dry forest': '#A4C639', # Light green 90EE90   
    'Plantation': '#F5DEB3', #  yellow
    'Bauxite extraction / quarry': '#B22222', #Iron Oxide Red
    'Water body': '#4682B4',  # Steel Blue
    'Buildings and other infrastructure': '#000000',  # Black
    'Mixed land use: forests with bamboo or agriculture/plantation': '#32CD32',  # lime Green
    'Mixed land use: agriculture and bamboo': '#D2B48C',  #light brown
    'Swamp forest': '#6B8E23',  # Olive Drab
    'Bamboo': '#F4A460',  # Sandy Brown
    'Forest': '#006400',  # Dark Green
}


# Get the list of all categories
all_categories = area_summary['Land Use Category'].tolist()

# Categories without custom colors
remaining_categories = [cat for cat in all_categories if cat not in custom_colors]

# Create a colormap for remaining categories
#cmap = plt.cm.get_cmap('Set3', len(remaining_categories))
cmap = plt.colormaps['Set3'](len(remaining_categories))

# Assign colors to remaining categories
colormap_colors = {}
for idx, category in enumerate(remaining_categories):
    color = mcolors.rgb2hex(cmap(idx))
    colormap_colors[category] = color

# Combine custom colors with colormap colors
category_colors = {**custom_colors, **colormap_colors}

# Map colors to the GeoDataFrame
terrestrial_landcover['color'] = terrestrial_landcover['Classify'].map(category_colors)


# Plot the GeoDataFrame with the assigned colors
fig, ax = plt.subplots(figsize=(18, 14), dpi=300) # Larger map size
terrestrial_landcover.plot(
    ax=ax,
    color=terrestrial_landcover['color'],
    linewidth=0.1,
    edgecolor='black'
)

# Create a base plot for the Jamaica boundary
fig, ax = plt.subplots(figsize=(10, 10))
jamaica_boundary.plot(ax=ax, color="none", edgecolor="black", linewidth=1, label="Jamaica Boundary")

# Overlay the landcover and bauxite reserves
landcover_bauxite.plot(ax=ax, color="grey", edgecolor="black", alpha=0.7, label="Bauxite Reserves")


def add_scale_bar(ax, length_km=20, location=(0.9, 0.8), linewidth=2, tick_height=0.01, label_offset=0.02, km_offset=0.03):
    """
    Add a scale bar to a plot, with "20" under the last tick mark and "km" positioned slightly to the right of the scale bar.
    """
    x, y = location  # Adjusted location (higher y value to move the scale bar upward)
    bar_half_length = 0.05  # Half the scale bar length in axes fraction

    # Draw the scale bar
    ax.plot(
        [x - bar_half_length, x + bar_half_length], [y, y],  # Scale bar endpoints
        transform=ax.transAxes, color='black', linewidth=linewidth
    )

    # Draw perpendicular tick marks
    tick_positions = [x - bar_half_length, x, x + bar_half_length]
    for pos in tick_positions:
        ax.plot(
            [pos, pos], [y - tick_height / 2, y + tick_height / 2],  # Vertical line for ticks
            transform=ax.transAxes, color='black', linewidth=linewidth
        )

    # Add numeric labels below the tick marks
    ax.text(
        x - bar_half_length, y - tick_height - label_offset, "0", transform=ax.transAxes, 
        ha='center', va='center', fontsize=10
    )
    ax.text(
        x, y - tick_height - label_offset, f"{length_km // 2}", transform=ax.transAxes, 
        ha='center', va='center', fontsize=10
    )
    ax.text(
        x + bar_half_length, y - tick_height - label_offset, f"{length_km}", transform=ax.transAxes, 
        ha='center', va='center', fontsize=10
    )

    # Add "km" label slightly to the right of the scale bar
    ax.text(
        x + bar_half_length + km_offset, y, "km", transform=ax.transAxes, 
        ha='left', va='center', fontsize=12
    )

def add_north_arrow(ax, location=(0.9, 0.8), size=0.05, fontsize=12, label_offset=0.03):
    """
    Add a north arrow to the plot, with "N" positioned slightly above the arrow.
    """
    x, y = location

    # Draw the arrow
    ax.annotate(
        '', xy=(x, y + size), xycoords='axes fraction',
        xytext=(x, y), textcoords='axes fraction',
        arrowprops=dict(facecolor='black', edgecolor='black', headwidth=10, headlength=15, width=5)
    )

    # Add the "N" label slightly above the arrow
    ax.text(
        x, y + size + label_offset, "N", transform=ax.transAxes,
        fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black"
    )
    
# Add the scale bar closer to the north arrow
add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, label_offset=0.04, km_offset=0.01)

# Add the north arrow
add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03)


plt.title(
    "Landcover on Bauxite Reserves", 
    fontsize=20,  # Larger font size for the title
    fontweight='bold',  # Bold font
    fontname='Times New Roman',  # Use Times New Roman font
    loc='center',  # Center the title
    pad=20  # Add some padding between the title and the map
)


In [ ]:
# Map colors to the landcover_bauxite GeoDataFrame
landcover_bauxite['color'] = landcover_bauxite['Classify'].map(category_colors)

# Plot the results
fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot land cover on bauxite reserves
landcover_bauxite.plot(
    ax=ax,
    color=landcover_bauxite['color'],
    edgecolor='black',
    linewidth=0.1
)

# Plot Jamaica boundary for reference
jamaica_boundary.plot(
    ax=ax,
    facecolor='none',
    edgecolor='black',
    linewidth=1,
    linestyle='--'
)

# Prepare legend handles
legend_handles = []
for _, row in bauxite_area_summary.iterrows():
    land_use = row['Land Use Type']
    color = category_colors[land_use]
    percentage = row['Percentage (%)']
    label = f"{land_use} ({percentage:.2f}%)"
    patch = mpatches.Patch(color=color, label=label)
    legend_handles.append(patch)

# Add legend below the plot
legend = ax.legend(
    handles=legend_handles,
    title="Land Use on Bauxite Reserves",
    bbox_to_anchor=(0.5, -0.1),
    loc='upper center',
    ncol=3,
    frameon=False,
    fontsize=12,
    title_fontsize=14,
    labelspacing=1.0,
    prop={'family': 'Times New Roman'}
)

# Add north arrow and scale bar
add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03)
add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, label_offset=0.04, km_offset=0.01)

# Add title
plt.title(
    "Landcover on Bauxite Reserves",
    fontsize=20,
    fontweight='bold',
    fontname='Times New Roman',
    loc='center',
    pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
# Rename or drop duplicate columns to avoid conflicts
protected_bauxite = protected_bauxite.rename(columns={"Shape_Area": "Bauxite_Shape_Area"})
terrestrial_landcover = terrestrial_landcover.rename(columns={"Shape_Area": "Landcover_Shape_Area"})

# Intersect bauxite reserves with protected areas
protected_bauxite = gpd.overlay(bauxite_reserves, combined_protected_layers, how='intersection')

# Intersect the result with landcover to get protected landcover on bauxite reserves
protected_landcover_bauxite = gpd.overlay(terrestrial_landcover, protected_bauxite, how='intersection')



In [ ]:
# Calculate the area of each intersected polygon
protected_landcover_bauxite['area_m2'] = protected_landcover_bauxite.geometry.area

# Aggregate by land use type
protected_bauxite_summary = (
    protected_landcover_bauxite.groupby('Classify')['area_m2']
    .sum()
    .reset_index()
)

# Calculate total protected area on bauxite reserves
total_protected_bauxite_area = protected_bauxite_summary['area_m2'].sum()

# Calculate percentage of each land use type under protected bauxite reserves
protected_bauxite_summary['percentage'] = (
    protected_bauxite_summary['area_m2'] / total_protected_bauxite_area
) * 100

# Add area in km²
protected_bauxite_summary['area_km2'] = protected_bauxite_summary['area_m2'] / 1e6

# Rename columns for clarity
protected_bauxite_summary.columns = ['Land Use Type', 'Area (m²)', 'Percentage (%)', 'Area (km²)']

# Display summary
display(protected_bauxite_summary)

# Map colors to the protected_landcover_bauxite GeoDataFrame
protected_landcover_bauxite['color'] = protected_landcover_bauxite['Classify'].map(category_colors)

# Plot the results
fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot protected landcover on bauxite reserves
protected_landcover_bauxite.plot(
    ax=ax,
    color=protected_landcover_bauxite['color'],
    edgecolor='black',
    linewidth=0.1
)

# Plot Jamaica boundary for reference
jamaica_boundary.plot(
    ax=ax,
    facecolor='none',
    edgecolor='black',
    linewidth=1,
    linestyle='--'
)

# Prepare legend handles
legend_handles = []
for _, row in protected_bauxite_summary.iterrows():
    land_use = row['Land Use Type']
    color = category_colors[land_use]
    percentage = row['Percentage (%)']
    label = f"{land_use} ({percentage:.2f}%)"
    patch = mpatches.Patch(color=color, label=label)
    legend_handles.append(patch)

# Add legend below the plot
legend = ax.legend(
    handles=legend_handles,
    title="Protected Land Use on Bauxite Reserves",
    bbox_to_anchor=(0.5, -0.1),
    loc='upper center',
    ncol=3,
    frameon=False,
    fontsize=12,
    title_fontsize=14,
    labelspacing=1.0,
    prop={'family': 'Times New Roman'}
)

# Add north arrow and scale bar
add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03)
add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, label_offset=0.04, km_offset=0.01)

# Add title
plt.title(
    "Protected Landcover on Bauxite Reserves",
    fontsize=20,
    fontweight='bold',
    fontname='Times New Roman',
    loc='center',
    pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
# Calculate the area of each intersected polygon for protected landcover on bauxite reserves
protected_landcover_bauxite['area_m2'] = protected_landcover_bauxite.geometry.area

# Aggregate by land use type for protected areas
protected_bauxite_summary = (
    protected_landcover_bauxite.groupby('Classify')['area_m2']
    .sum()
    .reset_index()
)

# Rename columns for clarity
protected_bauxite_summary.columns = ['Land Use Type', 'Protected Area (m²)']

# Add a column for percentage of protected land use types
protected_bauxite_summary['Percentage Protected (%)'] = (
    protected_bauxite_summary['Protected Area (m²)'] /
    protected_bauxite_summary['Protected Area (m²)'].sum()
) * 100

# Calculate all land use on bauxite reserves (both protected and not protected)
bauxite_landcover = gpd.overlay(terrestrial_landcover, bauxite_reserves, how='intersection')
bauxite_landcover['area_m2'] = bauxite_landcover.geometry.area

# Aggregate by land use type for all bauxite landcover
bauxite_landcover_summary = (
    bauxite_landcover.groupby('Classify')['area_m2']
    .sum()
    .reset_index()
)

# Rename columns for clarity
bauxite_landcover_summary.columns = ['Land Use Type', 'Total Area on Bauxite (m²)']

# Merge protected and total land use summaries
final_bauxite_summary = bauxite_landcover_summary.merge(
    protected_bauxite_summary,
    on='Land Use Type',
    how='left'
)

# Fill NaN values for unprotected land use with 0
final_bauxite_summary['Protected Area (m²)'] = final_bauxite_summary['Protected Area (m²)'].fillna(0)
final_bauxite_summary['Percentage Protected (%)'] = final_bauxite_summary['Percentage Protected (%)'].fillna(0)

# Add total area in km²
final_bauxite_summary['Total Area on Bauxite (km²)'] = final_bauxite_summary['Total Area on Bauxite (m²)'] / 1e6
final_bauxite_summary['Protected Area (km²)'] = final_bauxite_summary['Protected Area (m²)'] / 1e6

# Add percentage of each land use on bauxite reserves
final_bauxite_summary['Percentage on Bauxite (%)'] = (
    final_bauxite_summary['Total Area on Bauxite (m²)'] /
    final_bauxite_summary['Total Area on Bauxite (m²)'].sum()
) * 100

# Reorder columns for clarity
final_bauxite_summary = final_bauxite_summary[[
    'Land Use Type',
    'Total Area on Bauxite (km²)',
    'Protected Area (km²)',
    'Percentage Protected (%)',
    'Percentage on Bauxite (%)'
]]

# Display the summary table
display(final_bauxite_summary)

In [ ]:
# Calculate the area of each intersected polygon
protected_landcover_bauxite['area_m2'] = protected_landcover_bauxite.geometry.area

# Aggregate by land use type
protected_bauxite_summary = (
    protected_landcover_bauxite.groupby('Classify')['area_m2']
    .sum()
    .reset_index()
)

# Calculate total protected area on bauxite reserves
total_protected_bauxite_area = protected_bauxite_summary['area_m2'].sum()

# Calculate percentage of each land use type under protected bauxite reserves
protected_bauxite_summary['percentage'] = (
    protected_bauxite_summary['area_m2'] / total_protected_bauxite_area
) * 100

# Add area in km²
protected_bauxite_summary['area_km2'] = protected_bauxite_summary['area_m2'] / 1e6

# Rename columns for clarity
protected_bauxite_summary.columns = ['Land Use Type', 'Area (m²)', 'Percentage (%)', 'Area (km²)']

# Debugging Step: Check the column names to ensure the column is present
print(protected_bauxite_summary.columns)

# Display summary
display(protected_bauxite_summary)

# Map colors to the protected_landcover_bauxite GeoDataFrame
protected_landcover_bauxite['color'] = protected_landcover_bauxite['Classify'].map(category_colors)

# Plot the results
fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot the rest of the bauxite reserves as a transparent grey background
bauxite_reserves.plot(
    ax=ax,
    facecolor='grey',  # Transparent grey fill
    edgecolor='none',  # No edge for background
    alpha=0.3,  # Transparency level
    label="Remaining Bauxite Reserves"
)

# Plot protected landcover on bauxite reserves
protected_landcover_bauxite.plot(
    ax=ax,
    color=protected_landcover_bauxite['color'],
    edgecolor='black',
    linewidth=0.1,
    label="Protected Landcover on Bauxite Reserves"
)

# Plot Jamaica boundary for reference
jamaica_boundary.plot(
    ax=ax,
    facecolor='none',
    edgecolor='black',
    linewidth=1,
    linestyle='--'
)

# Prepare legend handles
legend_handles = [
    mpatches.Patch(facecolor='grey', alpha=0.3, label="Remaining Bauxite Reserves")  # Transparent grey patch
]

# Ensure the legend correctly handles missing or incorrect percentages
for _, row in protected_bauxite_summary.iterrows():
    land_use = row['Land Use Type']
    color = category_colors.get(land_use, '#CCCCCC')  # Default color if missing
    percentage = row.get('Percentage (%)', 0)  # Default to 0 if missing
    label = f"{land_use} ({percentage:.2f}%)"
    patch = mpatches.Patch(color=color, label=label)
    legend_handles.append(patch)

# Add legend below the plot
legend = ax.legend(
    handles=legend_handles,
    title="Protected Land Use on Bauxite Reserves",
    bbox_to_anchor=(0.5, -0.1),
    loc='upper center',
    ncol=3,
    frameon=False,
    fontsize=12,
    title_fontsize=14,
    labelspacing=1.0,
    prop={'family': 'Times New Roman'}
)

# Add north arrow and scale bar
add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03)
add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, label_offset=0.04, km_offset=0.01)

# Add title
plt.title(
    "Protected Landcover on Bauxite Reserves",
    fontsize=20,
    fontweight='bold',
    fontname='Times New Roman',
    loc='center',
    pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
# Each land use type’s percentage in the legend will represent the proportion of its protected area relative to its 
# total area on bauxite reserves.

### Invasive species (bamboo)

In [ ]:
# Area of bamboo
# Areas at risk near bamboo